# Convolutional Networks


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*,
unless another source is cited. Code cells adapted from the UDL notebooks (MIT
license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Outline

**Convolutional networks**

- Networks for images
- Invariance and equivariance
- 1D convolution
- Convolutional layers
- Channels
- Receptive fields
- Convolutional network for MNIST-1D


**2D convolutional networks**

- 2D convolution
- Downsampling and upsampling, 1x1 convolution
- Image classification: AlexNet and VGG
- Object detection
- Semantic segmentation
- Convolutions today


# Networks for images

Many of the problems we want to solve take images as input. We start with three
such tasks, then look at why fully-connected networks are a poor fit for them.


## Three tasks on images

![Image classification: one label for the whole image.](figs/07-Convolutional-Networks/image-classification-pipeline.png)

![Semantic segmentation: one label per pixel.](figs/07-Convolutional-Networks/image-segmentation-pipeline.png)


![Object detection: a box and a label per object.](figs/07-Convolutional-Networks/object-detection-examples.png)

- **Classification**: multiclass problem, one output per class. Convolutional network.
- **Detection**: classification plus box regression, at every location.
- **Segmentation**: multivariate binary classification, one output per pixel. Convolutional encoder-decoder.


## Why not a fully-connected network?

1. **Size**
    - 224x224 RGB image = 150,528 dimensions
    - Hidden layers generally larger than inputs
    - One hidden layer = 150,528 x 150,528 weights -- about 22.7 billion
2. **Nearby pixels are statistically related**
    - But a fully-connected network has no idea which inputs are neighbors: permute the pixels, retrain, and it does equally well
3. **Should be stable under transformations**
    - Don't want to re-learn the appearance of an object at every position in the image


**Convolutional networks**: parameters only look at local image patches, and the same parameters are shared across the image.


# Invariance and equivariance

Two properties describe how a network should respond when its input image is
transformed, for example translated. Classification wants the first,
segmentation the second, and convolution gives us a way to build both in.


## Invariance

- A function $\mathbf{f}[\mathbf{x}]$ is **invariant** to a transformation $\mathbf{t}[\,]$ if:

$$
\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{f}[\mathbf{x}]
$$

i.e., the function output is the same even after the transformation is applied.

e.g., image classification: the image has been translated, but we want our classifier to give the same result

![](figs/07-Convolutional-Networks/invariance-translated-image.png)


## Equivariance

- A function $\mathbf{f}[\mathbf{x}]$ is **equivariant** to a transformation $\mathbf{t}[\,]$ if:

$$
\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{t}[\mathbf{f}[\mathbf{x}]]
$$

i.e., the output is transformed in the same way as the input

e.g., image segmentation: the image has been translated and we want the segmentation to translate with it

![](figs/07-Convolutional-Networks/equivariance-segmentation.png)


# 1D convolution

Convolution builds both ideas in: the same small set of weights is applied at
every position of the input, so a shift of the input shifts the output.


## Convolution\* in 1D

- Input vector $\mathbf{x}$:

$$
\mathbf{x} = [x_1, x_2, \ldots, x_I]
$$

- Output is weighted sum of neighbors:

$$
z_i = \omega_1 x_{i-1} + \omega_2 x_i + \omega_3 x_{i+1}
$$

- Convolutional **kernel** or **filter**:

$$
\boldsymbol{\omega} = [\omega_1, \omega_2, \omega_3]^T
$$

**Kernel size** = 3

\* *Not technically convolution because weights order is not reversed*

## Convolution with kernel size 3

![](figs/07-Convolutional-Networks/conv1d-panel-a.png)


![](figs/07-Convolutional-Networks/conv1d-panel-b.png)


![](figs/07-Convolutional-Networks/conv1d-panel-c-zero-padding.png)


![](figs/07-Convolutional-Networks/conv1d-panel-d-valid.png)


- a, b) The same three weights at every position: **equivariant** to translation of the input, $\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{t}[\mathbf{f}[\mathbf{x}]]$.
- c) **Zero padding**: treat positions beyond the ends of the input as zero. Output has the same length as the input.
- d) **"Valid" convolution**: only process positions where the kernel falls inside the input. Output is smaller.

## Stride, kernel size, and dilation

- **Stride** = shift by k positions for each output
  - Decreases size of output relative to input
- **Kernel size** = weight a different number of inputs for each output
  - Combine information from a larger area
  - But kernel size 5 uses 5 parameters
- **Dilated** or **atrous** convolutions = intersperse kernel values with zeros
  - Combine information from a larger area
  - Fewer parameters

## Stride, kernel size, and dilation examples

![](figs/07-Convolutional-Networks/stride-panel-a.png)


![](figs/07-Convolutional-Networks/stride-panel-b.png)


![](figs/07-Convolutional-Networks/kernel-size-panel-c.png)


![](figs/07-Convolutional-Networks/dilation-panel-d.png)


a, b) size 3, stride 2; c) size 5, stride 1; d) size 3, stride 1, dilation 2

## 1D convolution example

```python
# Define a signal that we can apply convolution to
x = [5.2, 5.3, 5.4, 5.1, 10.1, 10.3, 9.9, 10.3, 3.2, 3.4, 3.3, 3.1]
```

In [ ]:
#| echo: false
#| fig-alt: "Two plots of the 12-sample step signal before (black) and after (red) convolution. Left, omega = [0.33, 0.33, 0.33]: the output is a smoothed version of the steps, pulled down at both ends by the zero padding. Right, omega = [-0.5, 0, 0.5]: the output is near zero on flat parts, positive where the signal rises and negative where it falls."
# Adapted from UDL notebook 10.1 (MIT)
import numpy as np
import matplotlib.pyplot as plt

x = np.array([5.2, 5.3, 5.4, 5.1, 10.1, 10.3, 9.9, 10.3, 3.2, 3.4, 3.3, 3.1])

fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, omega in zip(axs, ([0.33, 0.33, 0.33], [-0.5, 0, 0.5])):
    z = np.correlate(np.pad(x, 1), omega, mode="valid")   # zero padding, no kernel flip
    ax.plot(x, "k-", label="before")
    ax.plot(z, "r-", label="after")
    ax.axhline(0, color="#999999", lw=0.5)
    ax.set_title(f"omega = {omega}", fontsize=11, family="monospace")
    ax.set_xlim(0, 11)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

`length=3, stride=1, dilation=1, zero padding`

The general 1D convolution in a few lines: kernel size, stride, dilation and
zero padding or valid. The output length follows from where the kernel's centre
can go. The checks are the sums from UDL notebook 10.1, plus a comparison with
`np.convolve`, which flips the kernel (true convolution), so it matches only
after we reverse $\boldsymbol{\omega}$: that is the asterisk on the earlier
slide. The last lines check translation equivariance. With stride 1, shifting
the input shifts the output by the same amount. With stride 2, a shift of two
moves the output by one position, but a shift of one gives a different output,
not a shifted copy.

In [ ]:
# Adapted from UDL notebook 10.1 (MIT)
def conv1d(x, omega, stride=1, dilation=1, padding="zero"):
    """z_i = sum_k omega_k x[c_i + (k - (K-1)/2) * dilation], c_i the centre of output i (K odd)."""
    x, omega = np.asarray(x, float), np.asarray(omega, float)
    K = len(omega)
    reach = dilation * (K - 1) // 2          # how far the kernel extends each side of its centre
    if padding == "zero":
        x = np.pad(x, reach)                 # zeros beyond both ends
    centres = range(reach, len(x) - reach, stride)
    return np.array([sum(omega[k] * x[c + (k - (K - 1) // 2) * dilation] for k in range(K))
                     for c in centres])

avg3 = [0.33, 0.33, 0.33]
for label, omega, settings, udl_sum in [
        ("size 3, stride 1, dilation 1", avg3, {}, 71.1),
        ("size 3, stride 2, dilation 1", avg3, {"stride": 2}, None),
        ("size 5, stride 1, dilation 1", [0.2] * 5, {}, 69.6),
        ("size 3, stride 1, dilation 2", avg3, {"dilation": 2}, 68.3),
        ("size 3, stride 1, valid     ", avg3, {"padding": "valid"}, None)]:
    z = conv1d(x, omega, **settings)
    check = f" (UDL 10.1: {udl_sum})" if udl_sum else ""
    print(f"{label}: {len(x)} inputs -> {len(z):2d} outputs, sum = {z.sum():.1f}{check}")
print("stride 2 = every other stride-1 output:",
      np.allclose(conv1d(x, avg3, stride=2), conv1d(x, avg3)[::2]))

omega = np.array([-1.0, 0.5, -0.2])
print("matches np.convolve with the kernel as given:   ",
      np.allclose(conv1d(x, omega), np.convolve(x, omega, mode="same")))
print("matches np.convolve with the kernel reversed:   ",
      np.allclose(conv1d(x, omega), np.convolve(x, omega[::-1], mode="same")),
      "(valid:", np.allclose(conv1d(x, omega, padding="valid"),
                             np.convolve(x, omega[::-1], mode="valid")), end=")\n")

def shift(v, s):
    """Translate v right by s positions, filling with zeros."""
    return np.concatenate([np.zeros(s), v[:len(v) - s]])

rng = np.random.default_rng(0)
bump = np.pad(rng.normal(size=10), 8)        # zero near both ends, so nothing falls off
for stride, s in [(1, 1), (1, 3), (2, 2), (2, 1)]:
    lhs = conv1d(shift(bump, s), omega, stride=stride)      # f[t[x]]
    if s % stride == 0:
        rhs = shift(conv1d(bump, omega, stride=stride), s // stride)   # t[f[x]]
        print(f"stride {stride}, shift input by {s}: f[t[x]] = output shifted by {s // stride}?",
              np.allclose(lhs, rhs))
    else:
        base = conv1d(bump, omega, stride=stride)
        same = any(np.allclose(lhs, shift(base, k)) for k in range(len(base)))
        print(f"stride {stride}, shift input by {s}: f[t[x]] is a shift of f[x]? {same}")

# Convolutional layers

A convolutional layer adds a bias to the convolution and passes the result
through an activation function, just as a fully-connected layer does. The
difference is in how many parameters it has.


## Convolutional layer: output at position $i$

*(size: 3, stride: 1, dilation: 1)*

$$
\begin{aligned}
h_i &= \mathrm{a}\left[\beta + \omega_1 x_{i-1} + \omega_2 x_i + \omega_3 x_{i+1}\right] \\
&= \mathrm{a}\left[\beta + \sum_{j=1}^{3} \omega_j x_{i+j-2}\right]
\end{aligned}
$$

Convolutional layer: **3 weights, 1 bias** shared by all $i = 1 \ldots D$


Compare a fully-connected layer with $D$ inputs and $D$ hidden units:

$$
h_i = \mathrm{a}\left[\beta_i + \sum_{j=1}^{D} \omega_{ij} x_j\right]
$$

Fully-connected layer: **$D^2$ weights, $D$ biases**


## Convolution is a special case of a fully-connected layer

![](figs/07-Convolutional-Networks/fc-network-weights.png)

Fully connected network

6 inputs to each hidden unit


![](figs/07-Convolutional-Networks/conv-stride1-weights.png)

Convolution, size 3, stride 1, dilation 1, zero padding

3 inputs to each hidden unit


![](figs/07-Convolutional-Networks/conv-stride2-weights.png)

Convolution, size 3, stride 2, dilation 1, zero padding


Weight matrices. Bias is implied.

The same convolution as a matrix. Row $i$ holds the kernel, centred on column
$i \cdot \text{stride}$, with taps `dilation` columns apart; taps that land
beyond the input are the zero padding and are simply dropped. Multiplying by
this matrix gives exactly the output of `conv1d`. The matrix has $D^2$ entries
but only $K$ distinct values, which is the parameter saving.

In [ ]:
# Adapted from UDL notebook 10.1 (MIT)
def conv_matrix(n_in, omega, stride=1, dilation=1):
    """Weight matrix of a zero-padded 1D convolution (K odd)."""
    K = len(omega)
    n_out = -(-n_in // stride)                # ceil(n_in / stride)
    W = np.zeros((n_out, n_in))
    for i in range(n_out):
        for k in range(K):
            j = i * stride + (k - (K - 1) // 2) * dilation
            if 0 <= j < n_in:                 # outside the input: zero padding
                W[i, j] = omega[k]
    return W

np.set_printoptions(precision=2, suppress=True, linewidth=100)
omega = np.array([-1.0, 0.5, -0.2])
print("size 3, stride 1 (panel d):\n", conv_matrix(6, omega))
print("size 3, stride 2 (panel f):\n", conv_matrix(6, omega, stride=2))

for stride, dilation in [(1, 1), (2, 1), (1, 2)]:
    W = conv_matrix(len(x), omega, stride, dilation)
    print(f"stride {stride}, dilation {dilation}: W @ x equals conv1d(x)?",
          np.allclose(W @ x, conv1d(x, omega, stride, dilation)),
          f"| {W.size} matrix entries, {len(omega)} parameters")

## Read the convolution off the weight matrix

For each network: kernel size? stride? dilation? zero padding or valid? *(Bias is implied.)*

![1.](figs/07-Convolutional-Networks/question1-network.png)


![2.](figs/07-Convolutional-Networks/question2-network.png)


![3.](figs/07-Convolutional-Networks/question3-network.png)


Work these out before reading on. The number of distinct weights per row is
the kernel size, the gap between the non-zero entries is the dilation, the
shift from one row to the next is the stride, and a matrix with fewer rows
than columns (and no cut-off bands at the edges) is a valid convolution.



# Channels

A single convolution loses information, so convolutional layers compute several
convolutions in parallel and stack their outputs. This is where most of the
parameters come from.


## Channels

- The convolutional operation averages together the inputs, then passes the result through a ReLU
- Result is loss of information
- Solution: apply several convolutions and stack them in **channels** (also called **feature maps**)

![](figs/07-Convolutional-Networks/two-output-channels-a.png)


![](figs/07-Convolutional-Networks/two-output-channels-b.png)


Two output channels from one input channel: each channel has its own kernel and bias.

## Two input channels, one output channel

![](figs/07-Convolutional-Networks/two-input-channels.png)

The kernel now spans all input channels: $C_i \times K$ weights for one output channel.

## How many parameters?

- If there are $C_i$ input channels and kernel size $K$

$$
\boldsymbol{\Omega} \in \mathbb{R}^{C_i \times K} \qquad\qquad \boldsymbol{\beta} \in \mathbb{R}
$$

- If there are $C_i$ input channels and $C_o$ output channels

$$
\boldsymbol{\Omega} \in \mathbb{R}^{C_i \times C_o \times K} \qquad\qquad \boldsymbol{\beta} \in \mathbb{R}^{C_o}
$$

Total: $C_i \cdot C_o \cdot K + C_o$ parameters, **independent of the input length**.


# Receptive fields

The receptive field of a hidden unit is the region of the input that can
influence it. It depends on the kernel sizes, strides and dilations, not the
number of channels, and it grows as layers are stacked.


## Receptive fields

$\mathbb{R}^{C_i \times C_o \times K}$


![](figs/07-Convolutional-Networks/receptive-field-three-channels.png)


Indicates how many neighboring pixels influence the current output.

For one layer the receptive field depends only on the filter size (its effective size $d(K-1)+1$ with dilation $d$), not the number of channels

Each channel has a different set of filter weights.


## Receptive fields grow with depth


![](figs/07-Convolutional-Networks/receptive-field-four-layers.png)


Influence compounds in subsequent layers: the area of support is that of the convolution of the kernels.

$$
\begin{aligned}
[\omega_1, \omega_2, \omega_3] &\otimes [\omega_4, \omega_5, \omega_6] \\
&= [\omega_1', \ldots, \omega_5'] \\[4pt]
[\omega_1', \ldots, \omega_5'] &\otimes [\omega_1, \omega_2, \omega_3] \\
&= [\omega_1'', \ldots, \omega_7'']
\end{aligned}
$$

Two size-3 layers see 5 inputs, three see 7, each stride-1 layer adds $K-1$; a stride-2 layer doubles the growth of every layer after it. A dilated kernel counts with its effective size $d(K-1)+1$; a $k \times k$ pooling layer with stride $s$ counts as a kernel of size $k$, stride $s$.

Reminder: receptive field depends only on kernel size, stride and dilation, not on the number of channels.


Two ways to get the receptive field. The rule: each layer adds $(K-1)$ times
the product of the strides before it, with $K$ replaced by the effective size
$d(K-1)+1$ for a dilated kernel; a pooling layer of size $k$ and stride $s$ is
treated as a kernel of size $k$ and stride $s$. The brute-force check: multiply the
layers' weight matrices (all-ones kernels, one channel, since channels do not
change the answer) and count how many inputs reach one output unit. The first
stack is the figure (stride 2 at layer 3); the second is the MNIST-1D network
below, with valid convolutions, where the rule gives the same numbers.

In [ ]:
def receptive_fields(layers):
    """layers: list of (kernel size, stride). Returns the receptive field after each layer."""
    rf, jump, out = 1, 1, []
    for K, stride in layers:
        rf += (K - 1) * jump                  # K-1 more positions, at the current input spacing
        jump *= stride                        # a stride multiplies the spacing for later layers
        out.append(rf)
    return out

def receptive_fields_by_matrix(layers, n_in=64):
    M, n, out = np.eye(n_in), n_in, []
    for K, stride in layers:
        W = conv_matrix(n, np.ones(K), stride=stride)
        M, n = W @ M, W.shape[0]              # composite map from input to this layer
        out.append(int(np.count_nonzero(M[n // 2]))) # inputs reaching a unit in the middle
    return out

for name, layers in [("figure: K=3, strides 1, 1, 2, 1", [(3, 1), (3, 1), (3, 2), (3, 1)]),
                     ("MNIST-1D: K=3, stride 2, x3    ", [(3, 2)] * 3)]:
    print(f"{name}  rule: {receptive_fields(layers)}  "
          f"matrix product: {receptive_fields_by_matrix(layers)}")

# Convolutional network for MNIST-1D

We now compare a small convolutional network with a fully-connected network on
the MNIST-1D dataset, and count the parameters of each by hand.


## MNIST-1D dataset

![](figs/07-Convolutional-Networks/mnist1d-dataset.png)

## MNIST-1D results for fully-connected network

![](figs/07-Convolutional-Networks/mnist1d-fc-results.png)

Total parameters = 15,210 (two hidden layers of 100 units, 40-100-100-10)

## Convolutional network for MNIST-1D

![](figs/07-Convolutional-Networks/mnist1d-conv-network.png)

- Three convolutional layers (size 3, stride 2, 15 channels), then one fully-connected layer and a softmax
- Trained for 100,000 steps with SGD, LR = 0.01, batch size 100

## Counting the parameters

Use $C_i \cdot C_o \cdot K + C_o$ per layer:

- Conv 1: $1 \cdot 15 \cdot 3 + 15 = 60$
- Conv 2: $15 \cdot 15 \cdot 3 + 15 = 690$
- Conv 3: $15 \cdot 15 \cdot 3 + 15 = 690$
- Linear: $60 \cdot 10 + 10 = 610$
- **Total: 2,050** (vs. 150,185 for the fully connected network on the next slide)


Valid convolutions, size 3, stride 2: $40 \to 19 \to 9 \to 4$ positions ($\lfloor (D-3)/2 \rfloor + 1$ each), so the flattened input to the linear layer is $4 \times 15 = 60$.


```{.text style="font-size: 0.6em;"}
=====================================================================
Layer (type:depth-idx)            Output Shape         Param #
=====================================================================
Sequential                        [100, 10]            --
├─Conv1d: 1-1                     [100, 15, 19]        60
├─ReLU: 1-2                       [100, 15, 19]        --
├─Conv1d: 1-3                     [100, 15, 9]         690
├─ReLU: 1-4                       [100, 15, 9]         --
├─Conv1d: 1-5                     [100, 15, 4]         690
├─ReLU: 1-6                       [100, 15, 4]         --
├─Flatten: 1-7                    [100, 60]            --
├─Linear: 1-8                     [100, 10]            610
=====================================================================
Total params: 2,050
Trainable params: 2,050
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.07
=====================================================================
```


The same count as a calculator. Change the input length, channels, kernel size,
stride or padding and rerun. The output-length formula is the general one,
$\lfloor (D + 2p - d(K-1) - 1)/s \rfloor + 1$, which reduces to
$\lfloor (D-3)/2 \rfloor + 1$ for a valid, size 3, stride 2 convolution. The
last lines check the fully connected networks quoted on these slides.

In [ ]:
D = 40            # input length (MNIST-1D)
C = 15            # channels in every conv layer
K = 3             # kernel size
stride = 2
padding = 0       # 0 = valid; (K - 1) // 2 = zero padding that keeps the size at stride 1
n_conv = 3
n_classes = 10

def conv_out_len(D, K, stride=1, padding=0, dilation=1):
    return (D + 2 * padding - dilation * (K - 1) - 1) // stride + 1

def conv_params(C_in, C_out, K, dims=1):
    return C_in * C_out * K ** dims + C_out    # weights + one bias per output channel

def fc_params(widths):
    return sum(a * b + b for a, b in zip(widths[:-1], widths[1:]))

sizes, c_in, total = [D], 1, 0
for layer in range(1, n_conv + 1):
    sizes.append(conv_out_len(sizes[-1], K, stride, padding))
    p = conv_params(c_in, C, K)
    total += p
    print(f"Conv {layer}: {c_in:2d} x {C} x {K} + {C} = {p:4d} params, output {sizes[-1]} x {C}")
    c_in = C
p = fc_params([sizes[-1] * C, n_classes])
total += p
print(f"Linear: {sizes[-1] * C} x {n_classes} + {n_classes} = {p:4d} params")
print(f"Total: {total:,} params; positions {' -> '.join(map(str, sizes))}")

print(f"\nFully connected 40-100-100-10:     {fc_params([40, 100, 100, 10]):,} params")
print(f"Fully connected 40-285-135-60-10:  {fc_params([40, 285, 135, 60, 10]):,} params "
      f"({fc_params([40, 285, 135, 60, 10]) / total:.0f}x the CNN)")

## Performance, and why

![](figs/07-Convolutional-Networks/mnist1d-conv-vs-fc.png)

The figure (UDL Fig. 10.8b) is from an earlier draft of the book: the current
UDL text (§10.2.6) describes a 285/135/60 fully connected network with 59,065
parameters (29× the CNN).


The convolutional network has 70x fewer parameters and generalizes far better, because of its **inductive bias**:

- Forced to process each location the same way, so it shares what it learns across locations
- Searches a smaller family of input-output mappings, all of which are plausible for this data


# 2D convolution

The second part of the lecture moves from 1D signals to images, then to the
operations and architectures built from 2D convolutions. Everything from the 1D
case carries over; the kernel is just a small grid instead of a short vector.


## 2D convolution

- Weighted sum over a $K \times K$ region, so $K \times K$ weights
- Build into a convolutional layer by adding a bias and passing through an activation function

$$
h_{i,j} = \mathrm{a}\left[\beta + \sum_{m=1}^{3} \sum_{n=1}^{3} \omega_{m,n} x_{i+m-2,\, j+n-2}\right]
$$

![](figs/07-Convolutional-Networks/conv2d-panels-ab.png)

## 2D convolution with zero padding

![](figs/07-Convolutional-Networks/conv2d-zero-padding-panels-cd.png)

Kernel size, stride, dilation and valid convolution all work as in 1D, now in each spatial dimension.

## Channels and parameters in 2D

![](figs/07-Convolutional-Networks/conv2d-channels.png)

- $C_i$ input channels, kernel size $K \times K$, one output channel: $\boldsymbol{\Omega} \in \mathbb{R}^{C_i \times K \times K}$, $\beta \in \mathbb{R}$
- $C_i$ input channels, $C_o$ output channels: $\boldsymbol{\Omega} \in \mathbb{R}^{C_i \times C_o \times K \times K}$, $\boldsymbol{\beta} \in \mathbb{R}^{C_o}$

e.g., an RGB input to 64 channels with 3x3 kernels: $3 \cdot 64 \cdot 9 + 64 = 1{,}792$ parameters, whatever the image size.


A 2D convolutional layer from scratch, with weights
$\boldsymbol{\Omega} \in \mathbb{R}^{C_o \times C_i \times K \times K}$ (the
PyTorch order), applied to a synthetic image with three hand-set kernels: two
Sobel edge detectors and a box blur. Each output channel responds to a
different pattern; a trained network learns kernels like these (see the
AlexNet filters below). The printed lines check translation equivariance in 2D
and repeat the parameter counts from this slide and from the start of the
lecture.

In [ ]:
#| code-fold: true
#| fig-cap: "A 3x3 convolutional layer with one input and three output channels, zero padding, no activation. The edge detectors respond with opposite signs on the two sides of each shape."
#| fig-alt: "Four 32 by 32 images. Input: a white square at the upper left and a white disk at the lower right on black. Vertical edges: red along the left edges of both shapes and blue along their right edges, zero elsewhere. Horizontal edges: red along the top edges and blue along the bottom edges. Blur: the same two shapes with softened borders."
# Adapted from UDL notebook 10.3 (MIT)
def conv2d(image, weights, bias, stride=1, pad=1):
    """image: (C_i, H, W); weights: (C_o, C_i, K, K); bias: (C_o,). Zero padding, no flip."""
    image = np.pad(image, ((0, 0), (pad, pad), (pad, pad)))
    C_o, C_i, K, _ = weights.shape
    H_out = (image.shape[1] - K) // stride + 1
    W_out = (image.shape[2] - K) // stride + 1
    out = np.zeros((C_o, H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = image[:, i * stride:i * stride + K, j * stride:j * stride + K]
            out[:, i, j] = np.tensordot(weights, patch, axes=3) + bias   # sum over C_i, K, K
    return out

rows, cols = np.mgrid[0:32, 0:32]
square = (np.abs(rows - 10) < 5) & (np.abs(cols - 10) < 5)
disk = (rows - 21) ** 2 + (cols - 21) ** 2 < 36
image = (square | disk).astype(float)[None]               # one input channel

sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
weights = np.stack([sobel, sobel.T, np.full((3, 3), 1 / 9)])[:, None]   # (3, 1, 3, 3)
h = conv2d(image, weights, bias=np.zeros(3))

shifted = np.roll(image, (2, 3), axis=(1, 2))             # the shapes stay clear of the border
print("2D equivariance, shift (2, 3): f[t[x]] = t[f[x]]?",
      np.allclose(conv2d(shifted, weights, np.zeros(3)), np.roll(h, (2, 3), axis=(1, 2))))
print("stride 2 output shape:", conv2d(image, weights, np.zeros(3), stride=2).shape)
print(f"this layer: {conv_params(1, 3, 3, dims=2)} params;  "
      f"RGB -> 64 channels, 3x3: {conv_params(3, 64, 3, dims=2):,} params")
d = 224 * 224 * 3
print(f"one fully connected layer on 224x224 RGB: {d:,} x {d:,} = {d * d / 1e9:.1f} billion weights")

fig, axs = plt.subplots(1, 4, figsize=(10, 2.8))
panels = [(image[0], "input", "gray"), (h[0], "vertical edges", "RdBu_r"),
          (h[1], "horizontal edges", "RdBu_r"), (h[2], "blur", "gray")]
for ax, (im, title, cmap) in zip(axs, panels):
    lim = np.abs(im).max()
    ax.imshow(im, cmap=cmap, vmin=-lim if cmap == "RdBu_r" else 0, vmax=lim)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Downsampling and upsampling, 1x1 convolution

Convolutional networks for images change the spatial size of their
representations as they go, and mix information across channels. These are the
remaining building blocks before we look at full architectures.


## Downsampling

![](figs/07-Convolutional-Networks/downsample-panel-a.png)

Sample every other position (equivalent to stride two)


![](figs/07-Convolutional-Networks/downsample-panel-b-max-pool.png)

Max pooling (partial invariance to translation)


![](figs/07-Convolutional-Networks/downsample-panel-c-mean-pool.png)

Mean pooling


## Upsampling

![](figs/07-Convolutional-Networks/upsample-panel-a-duplicate.png)

Duplicate


![](figs/07-Convolutional-Networks/upsample-panel-b-max-unpool.png)

Max-unpooling

Uses same spatial locations from max pool. (`return_indices=True`)


![](figs/07-Convolutional-Networks/upsample-panel-c-bilinear.png)

Bilinear interpolation


## Transposed convolutions

![](figs/07-Convolutional-Networks/stride2-convolution.png)

Kernel size 3, stride 2 convolution


![](figs/07-Convolutional-Networks/transposed-convolution.png)

Transposed convolution: a learned upsampling


## 1x1 convolution

![](figs/07-Convolutional-Networks/one-by-one-convolution.png)

- Mixes channels
- Can change number of channels
- Equivalent to running same fully connected network at each position

# Image classification

Image classification on ImageNet is where deep convolutional networks first
showed their strength. AlexNet and VGG are the two architectures to know.


## ImageNet 1K database

![](figs/07-Convolutional-Networks/imagenet-examples.png)


![Fei-Fei Li](figs/07-Convolutional-Networks/fei-fei-li.png)

![](figs/07-Convolutional-Networks/the-worlds-i-see-book.png)


- 224 x 224 images
- 1,281,167 training images, 50,000 validation images, and 100,000 test images
- 1000 classes

## AlexNet (2012)

![](figs/07-Convolutional-Networks/alexnet-architecture.png)


Almost all the 60 million parameters are in fully connected layers


[A. Krizhevsky, I. Sutskever, and G. E. Hinton, "ImageNet classification with
deep convolutional neural networks," *Advances in Neural Information Processing
Systems 25* (NeurIPS 2012); reprinted in *Commun. ACM*, vol. 60, no. 6, pp. 84--90,
May 2017, doi: [10.1145/3065386](https://doi.org/10.1145/3065386).]{style="font-size: 0.7em;"}

## AlexNet: training and results

![Figure 2 of Krizhevsky et al.](figs/07-Convolutional-Networks/alexnet-paper-figure.png)

- Won the 2012 Large-Scale Visual Recognition Challenge (ILSVRC) by a big margin: **15.3% top-5** test error for the winning entry, an ensemble of seven CNNs, two pretrained on the full ImageNet 2011 release; five CNNs trained on the challenge data alone give 16.4% top-5 and 38.1% top-1.
- Took between five and six days to train on two GTX 580 3GB GPUs with manually optimized compute kernels.
- SGD with momentum 0.9, batch size 128, L2 (weight decay) regularization.
- At test time, average results from ten patches: five 224×224 crops (four corners and centre) and their horizontal reflections.

## AlexNet learned filters

![Cat image input (not actual image)](figs/07-Convolutional-Networks/alexnet-cat-input.png)


![Source: <https://cs231n.github.io/understanding-cnn/>](figs/07-Convolutional-Networks/alexnet-activations-filters.png)



## ~1959: Hubel & Wiesel -- Visual cortex and receptive fields

![](figs/07-Convolutional-Networks/hubel-wiesel-photo.png)

David Hubel and Torsten Wiesel discovered how neurons in the visual cortex
respond to specific patterns of light, such as edges and orientations. Their
work on receptive fields provided key insights into hierarchical processing in
vision, influencing the design of modern convolutional neural networks.


![](figs/07-Convolutional-Networks/hubel-wiesel-experiment.png)


![](figs/07-Convolutional-Networks/visual-pathway.png)


## AlexNet regularization: augmentation and dropout

![](figs/07-Convolutional-Networks/data-augmentation.png)

Data augmentation by a factor of 2048 using (i) spatial transformations and (ii) modifications of the input intensities.


![](figs/07-Convolutional-Networks/dropout.png)

Dropout was applied in the fully connected layers.


## VGG (2014; ICLR 2015)

![](figs/07-Convolutional-Networks/vgg-architecture.png)

- Only 3x3 convolutions and 2x2 max pooling: deeper, more regular
- 19 hidden layers, 144 million parameters
- 6.8% top-5 error rate, 23.7% top-1 error rate

[K. Simonyan and A. Zisserman, "Very deep convolutional networks for large-scale image recognition," ICLR 2015, [arXiv:1409.1556](https://arxiv.org/abs/1409.1556).]{style="font-size: 0.7em;"}

## ImageNet history

![](figs/07-Convolutional-Networks/imagenet-history.png)

# Object detection

Object detection asks the network to find each object and its class, not just
to label the whole image. The same convolutional backbone does the work; what
changes is the output layer.


## You Only Look Once (YOLO)

- Network similar to VGG (448x448 input)
- 7×7 grid of locations
- Predict class at each location
- Predict 2 bounding boxes at each location
  - Five parameters -- x, y, height, width, and confidence
- Momentum, weight decay, dropout, and data augmentation
- Heuristic at the end to threshold and decide final boxes -- (non-maximum suppression)


![](figs/07-Convolutional-Networks/yolo-grid.png)


[J. Redmon, S. Divvala, R. Girshick, and A. Farhadi, "You only look once: unified, real-time object detection," CVPR 2016, [arXiv:1506.02640](https://arxiv.org/abs/1506.02640).]{style="font-size: 0.7em;"}

## YOLO: from grid to boxes

![](figs/07-Convolutional-Networks/yolo-detection-steps.png)

## Transfer learning

![](figs/07-Convolutional-Networks/transfer-learning.png)

Detection and segmentation networks start from a backbone pretrained on ImageNet classification, then replace the output layers and fine-tune.

# Semantic segmentation

Semantic segmentation labels every pixel, so the network has to bring its
compressed representation back up to the full image size. That is what the
upsampling operations were for.


## Semantic segmentation (2015)

![](figs/07-Convolutional-Networks/segmentation-encoder-decoder.png)

- **Encoder** (VGG) downsamples to a compact representation; **decoder** mirrors it with max-unpooling and transposed convolutions back to full resolution; a 1x1 convolution and softmax give per-pixel class probabilities.
- The **U-Net** ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)) adds skip connections from each encoder level to the matching decoder level, so fine spatial detail does not have to squeeze through the bottleneck. It is the default segmentation architecture and appears again in diffusion models.

## Semantic segmentation results

![](figs/07-Convolutional-Networks/segmentation-results.png)

# Convolutions today

Transformers took over the ImageNet leaderboard after 2020, but convolutions did
not go away. They sit at the input of vision transformers and inside the
generators of diffusion models, and a modernized CNN keeps up with transformers.


## Where convolutions live in 2026

- **Patch embedding in vision transformers.** ViT ([Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929)) splits the image into 16x16 patches and linearly projects each one: that is a convolution with kernel size 16 and stride 16, and it is implemented as `nn.Conv2d` in most codebases. Lecture 16.
- **The U-Net in diffusion models.** DDPM ([Ho et al., 2020](https://arxiv.org/abs/2006.11239)) and Stable Diffusion ([Rombach et al., 2022](https://arxiv.org/abs/2112.10752)) denoise with a convolutional U-Net; later diffusion transformers still patchify the input the ViT way. Lecture 18.
- **ConvNeXt** ([Liu et al., 2022](https://arxiv.org/abs/2201.03545)): take a ResNet, apply the transformer-era training recipe and design choices (patchified stem, large 7x7 depthwise kernels, fewer activations and norms, LayerNorm, GELU), and the pure CNN matches or beats the Swin transformer at every model size on ImageNet, with the same accuracy-throughput trade-off.
- So the inductive bias of this lecture, locality and parameter sharing, is still what the data demands; the question is only how much of it to build in and how much to learn.


# Summary

What to take away from this lecture; the recap deck and the knowledge check
are built from this list.


## Summary

- Fully-connected layers on images are too large (150,528 x 150,528 weights for one 224x224 RGB hidden layer), ignore which pixels are neighbors, and must re-learn every object at every position.
- **Invariant**: $\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{f}[\mathbf{x}]$ (classification). **Equivariant**: $\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{t}[\mathbf{f}[\mathbf{x}]]$ (segmentation). Convolution is equivariant to translation.
- A 1D convolutional layer computes $h_i = \mathrm{a}[\beta + \sum_j \omega_j x_{i+j-2}]$ with one kernel shared by every position; kernel size, stride, dilation, and zero-padded vs. valid set the output size and area of support.
- Convolution is a fully-connected layer with a banded, repeated weight matrix: $K$ weights and 1 bias instead of $D^2$ and $D$.
- Channels: $C_i$ input and $C_o$ output channels with kernel size $K$ need $C_i \cdot C_o \cdot K + C_o$ parameters ($C_i \cdot C_o \cdot K \cdot K + C_o$ in 2D), independent of the input size.
- The receptive field grows with depth (two size-3 layers see 5 inputs, three see 7) and depends on kernel size, stride and dilation, not on the number of channels; a $k \times k$ pooling layer with stride $s$ acts as a kernel of size $k$, stride $s$.
- MNIST-1D: a 2,050-parameter convolutional network beats a 150,185-parameter fully-connected one because its inductive bias restricts the search to plausible mappings.
- Downsampling by stride, max or mean pooling; upsampling by duplication, max-unpooling, bilinear interpolation or transposed convolution; 1x1 convolution mixes channels.
- AlexNet (2012: 60M parameters, 16.4% top-5 for five CNNs, 15.3% for the winning seven-CNN entry) and VGG (2014: 3x3 convolutions throughout, 144M parameters, 6.8% top-5) set the ImageNet template; augmentation, dropout and weight decay made them train.
- Detection (YOLO: class and boxes per grid cell, then non-maximum suppression) and segmentation (encoder-decoder, U-Net) reuse an ImageNet-pretrained backbone with a new output layer.
- Convolutions survive in 2026 as the ViT patch embedding and the diffusion U-Net, and ConvNeXt shows a modernized CNN matches transformers on ImageNet.

## CNN Explainer

![<https://poloclub.github.io/cnn-explainer/>](figs/07-Convolutional-Networks/cnn-explainer.png)